In [ ]:
!nvidia-smi

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
%%capture
import os
if not os.path.exists('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-new_idea_2'):
    !unzip /content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-new_idea_2.zip

In [3]:
%cd /content/AAAI-TALAS-new_idea_2

In [4]:
# Remove stale cached teacher embeddings before training
!rm -rf cache
!mkdir -p cache
!echo "Removed cache/"

In [ ]:
!pip install -r requirements.txt

In [5]:
# HeatGeo ablation: disable pointwise teacher anchor and task contrastive loss.
!sed -i 's|\.\./main.py|main.py|g' scripts/train_heatgeo.sh
!sed -i 's|TRAIN_DATA=.*|TRAIN_DATA="data/merged_9_data_3k_each_ver2.csv"|g' scripts/train_heatgeo.sh
!sed -i 's|TEACHER_MODEL=.*|TEACHER_MODEL="Qwen/Qwen3-Embedding-4B"|g' scripts/train_heatgeo.sh
!sed -i 's|STUDENT_MODEL=.*|STUDENT_MODEL="google-bert/bert-base-uncased"|g' scripts/train_heatgeo.sh
!sed -i 's|SAVE_DIR=.*|SAVE_DIR="checkpoints/heatgeo_no_anchor_task"|g' scripts/train_heatgeo.sh
!sed -i -E 's|^[[:space:]]*w_task[[:space:]]*=.*|    w_task = 0.0|g' config/heatgeo_config.py
!sed -i -E 's|^[[:space:]]*lambda_anchor[[:space:]]*=.*|    lambda_anchor = 0.0|g' config/heatgeo_config.py
!sed -i -E 's|^[[:space:]]*diffusion_topk[[:space:]]*=.*|    diffusion_topk = 64|g' config/heatgeo_config.py
!sed -i -E 's|^[[:space:]]*hard_neg_k[[:space:]]*=.*|    hard_neg_k = 32|g' config/heatgeo_config.py
!sed -i -E 's|^[[:space:]]*random_neg_k[[:space:]]*=.*|    random_neg_k = 32|g' config/heatgeo_config.py
!sed -i -E 's|^[[:space:]]*candidate_size[[:space:]]*=.*|    candidate_size = 128|g' config/heatgeo_config.py
!grep -E "w_task|lambda_anchor|diffusion_topk|hard_neg_k|random_neg_k|candidate_size" config/heatgeo_config.py
!cat scripts/train_heatgeo.sh | grep -E "METHOD|TRAIN_DATA|TEACHER_MODEL|STUDENT_MODEL|SAVE_DIR"

!sed -i -E 's/BATCH_SIZE=[0-9]+/BATCH_SIZE=8/g' scripts/train_heatgeo.sh
!sed -i -E 's|--batch_size[[:space:]]+[0-9]+|--batch_size 8|g' scripts/train_heatgeo.sh

# Kiểm tra lại các dòng có chứa từ khoá batch trong script
!cat scripts/train_heatgeo.sh | grep -i "batch"
!test -f data/merged_9_data_3k_each_ver2.csv || (echo 'Missing data/merged_9_data_3k_each_ver2.csv. Re-run the unzip cell or check the archive.' && false)
!mkdir -p analysis/heatgeo_no_anchor_task_diagnostics
!bash -c 'set -o pipefail; bash scripts/train_heatgeo.sh 2>&1 | tee analysis/heatgeo_no_anchor_task_diagnostics/train_heatgeo_no_anchor_task.log'

In [ ]:
# Run all validation and test benchmarks from the saved HeatGeo no-anchor/no-task checkpoint
import re
from pathlib import Path

import torch
from transformers import AutoModel

from src.evaluation.evaluation_automodel import (
    eval_classification_task,
    eval_pair_task,
    eval_sts_task,
    eval_cls_tasks,
    eval_pair_tasks,
    eval_sts_tasks,
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

checkpoint_dir = Path("checkpoints/heatgeo_no_anchor_task")
best_checkpoint = checkpoint_dir / "best_model.pt"

if best_checkpoint.exists():
    checkpoint_path = best_checkpoint
else:
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint_epoch_*.pt"),
        key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.stem).group(1)),
    )
    assert checkpoints, f"No checkpoint found in {checkpoint_dir}"
    checkpoint_path = checkpoints[-1]

print(f"Loading checkpoint: {checkpoint_path}")

try:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = checkpoint.get("config", {})
student_model_name = cfg.get("student_model_name", "google-bert/bert-base-uncased")
print(f"Student model: {student_model_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModel.from_pretrained(student_model_name)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

benchmark_groups = [
    ("Validation classification", eval_classification_task, eval_cls_tasks),
    ("Validation pair classification", eval_pair_task, eval_pair_tasks),
    ("Validation STS", eval_sts_task, eval_sts_tasks),
    ("Test classification", eval_classification_task, test_cls_tasks),
    ("Test pair classification", eval_pair_task, test_pair_tasks),
    ("Test STS", eval_sts_task, test_sts_tasks),
]

for name, eval_fn, tasks in benchmark_groups:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    eval_fn(model, tasks)

print("\nAll benchmark evaluations finished.")